# Feature engineering for customer churn prediction

This notebook walks through the full feature engineering pipeline:
loading the generated telecom data, encoding categorical variables,
scaling numeric features, creating interaction terms, and selecting
the final feature set for modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from sklearn.preprocessing import StandardScaler, LabelEncoder

pd.set_option('display.max_columns', 40)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Load telecom data

The data was created by `generate_data.py` and saved to `data/telco_churn.csv`.
It contains 5,000 customers with realistic churn correlations.

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')
print(f'Shape: {df.shape}')
print(f'Churn rate: {(df["churn"] == "Yes").mean():.3f}')
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

In [ ]:
# Data types overview
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'Numeric columns ({len(num_cols)}): {num_cols}')

## 2. Handle missing values

The `total_charges` column has ~30 missing values (by design from generate_data.py).
We impute these as `monthly_charges * tenure_months`, which is the logical
relationship between these fields.

In [ ]:
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
missing_tc = df['total_charges'].isnull().sum()
print(f'Missing total_charges before imputation: {missing_tc}')

df['total_charges'].fillna(
    df['monthly_charges'] * df['tenure_months'], inplace=True
)
print(f'Missing total_charges after imputation:  {df["total_charges"].isnull().sum()}')

## 3. Encode categorical variables

We use two encoding strategies:
- **Binary columns** (Yes/No, Male/Female): label encoding (0/1)
- **Multi-class columns** (contract type, payment method, etc.): one-hot encoding

In [ ]:
# Drop customer_id (not a feature)
df_feat = df.drop(columns=['customer_id']).copy()

# Encode target
df_feat['churn'] = (df_feat['churn'] == 'Yes').astype(int)

# Binary columns: label encode
binary_cols = ['gender', 'partner', 'dependents', 'phone_service', 'paperless_billing']
le_dict = {}
for col in binary_cols:
    le = LabelEncoder()
    df_feat[col] = le.fit_transform(df_feat[col])
    le_dict[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
# Multi-class columns: one-hot encode
multi_cols = [
    'contract', 'internet_service', 'multiple_lines',
    'online_security', 'online_backup', 'device_protection',
    'tech_support', 'streaming_tv', 'streaming_movies',
    'payment_method',
]

print(f'Columns before one-hot encoding: {df_feat.shape[1]}')
df_feat = pd.get_dummies(df_feat, columns=multi_cols, drop_first=True)
print(f'Columns after one-hot encoding:  {df_feat.shape[1]}')
print(f'\nNew dummy columns sample:')
dummy_cols = [c for c in df_feat.columns if '_' in c and c not in binary_cols + ['customer_id']]
print(dummy_cols[:10])

## 4. Create interaction features

We engineer domain-informed features that capture known churn risk factors.

In [ ]:
# Tenure group
bins = [0, 12, 24, 48, 72]
labels = ['0-12', '13-24', '25-48', '49-72']
df_feat['tenure_group'] = pd.cut(
    df_feat['tenure_months'], bins=bins, labels=labels, include_lowest=True
)
df_feat = pd.get_dummies(df_feat, columns=['tenure_group'], drop_first=True)

# Charges per month of tenure
df_feat['charges_per_month_tenure'] = np.where(
    df_feat['tenure_months'] > 0,
    df_feat['total_charges'] / df_feat['tenure_months'],
    df_feat['monthly_charges']
)

# Count of support services
support_cols = [c for c in df_feat.columns if any(
    s in c for s in ['online_security_Yes', 'online_backup_Yes',
                     'device_protection_Yes', 'tech_support_Yes']
)]
if support_cols:
    df_feat['has_support_services'] = df_feat[support_cols].sum(axis=1)
else:
    # Fallback: look at original binary columns
    df_feat['has_support_services'] = 0

print(f'Final feature count: {df_feat.shape[1]}')
print(f'\nEngineered feature stats:')
df_feat[['charges_per_month_tenure', 'has_support_services']].describe().round(2)

## 5. Feature scaling with StandardScaler

Models like Logistic Regression require features on similar scales.
We fit the scaler on the feature matrix and show the before/after distributions.

In [ ]:
feature_cols = [c for c in df_feat.columns if c != 'churn']
X = df_feat[feature_cols].values.astype(float)
y = df_feat['churn'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Compare distributions before and after scaling for key numeric features
key_numeric = ['tenure_months', 'monthly_charges', 'total_charges', 'charges_per_month_tenure']
key_idx = [feature_cols.index(c) for c in key_numeric if c in feature_cols]

fig, axes = plt.subplots(2, len(key_idx), figsize=(14, 6))
for j, idx in enumerate(key_idx):
    axes[0, j].hist(X[:, idx], bins=30, alpha=0.7, color='steelblue')
    axes[0, j].set_title(f'{feature_cols[idx]} (raw)')
    axes[1, j].hist(X_scaled[:, idx], bins=30, alpha=0.7, color='coral')
    axes[1, j].set_title(f'{feature_cols[idx]} (scaled)')

plt.suptitle('Feature distributions before and after StandardScaler', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Correlation matrix

We examine feature correlations to identify redundancy and confirm
that our engineered features have the expected relationships with churn.

In [ ]:
# Correlation with churn target (top 20 features)
corr_with_churn = df_feat[feature_cols + ['churn']].corr()['churn'].drop('churn')
top_corr = corr_with_churn.abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#FF5722' if corr_with_churn[f] > 0 else '#2196F3'
          for f in top_corr.index]
ax.barh(range(len(top_corr)), [corr_with_churn[f] for f in top_corr.index],
        color=colors)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index)
ax.set_xlabel('Correlation with churn')
ax.set_title('Top 20 features by correlation with churn')
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Inter-feature correlation heatmap for top features
top_features = list(top_corr.index[:12]) + ['churn']
corr_matrix = df_feat[top_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation heatmap (top features + target)')
plt.tight_layout()
plt.show()

## 7. Feature selection rationale

We retain all engineered features for modeling because:

- **Contract type** dummies are the strongest churn predictors
  (month-to-month customers churn at much higher rates)
- **Tenure** and **tenure group** capture the nonlinear relationship
  between customer age and loyalty
- **Charges per month of tenure** normalizes spending across tenure
  lengths, providing a cleaner signal than raw total_charges
- **Support service count** aggregates protective factors into a
  single feature -- customers with more services are stickier
- **Payment method** dummies flag electronic check users who have
  notably higher churn

We do not apply aggressive feature reduction because the tree-based
models (RF, XGBoost, LightGBM) handle high-dimensional sparse features
well, and Logistic Regression is regularized.

In [ ]:
# Validate via the data_loader pipeline
from src.data_loader import load_and_prepare

X_train, X_test, y_train, y_test, feat_names = load_and_prepare(
    filepath='../data/telco_churn.csv'
)

print(f'\nTotal features from pipeline: {len(feat_names)}')
print(f'Feature names: {feat_names[:15]} ...')

## Summary

The feature engineering pipeline transforms 21 raw columns into a modeling-ready
matrix with ~35 features after one-hot encoding and interaction feature creation.
Contract type, tenure, and payment method are the strongest churn predictors.
The scaled feature set is ready for training in `03_modeling.ipynb`.